In [54]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

# -----------------------------
# Device
# -----------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------------
# Define Binary Model
# -----------------------------
class BinaryCancerModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = models.resnet18(pretrained=False)
        self.model.fc = nn.Linear(self.model.fc.in_features, 2)  # Binary

    def forward(self, x):
        return self.model(x)

# Load Binary model
binary_model = models.resnet18(pretrained=False)
binary_model.fc = nn.Linear(binary_model.fc.in_features, 2)  # Binary
binary_model.load_state_dict(
    torch.load("/kaggle/input/models/pytorch/default/1/breast_cancer_resnet18.pth", map_location=DEVICE)
)
binary_model.to(DEVICE).eval()

# -----------------------------
# Define TNM Model (matching checkpoint)
# -----------------------------
class TNMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.resnet18(pretrained=False)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.fc_T = nn.Linear(in_features, 3)  # checkpoint has 3 classes
        self.fc_N = nn.Linear(in_features, 3)  # checkpoint has 3 classes
        self.fc_M = nn.Linear(in_features, 2)  # M usually binary

    def forward(self, x):
        features = self.backbone(x)
        return self.fc_T(features), self.fc_N(features), self.fc_M(features)

# Load TNM model
tnm_model = TNMModel().to(DEVICE)
tnm_model.load_state_dict(
    torch.load("/kaggle/input/models/pytorch/default/1/tnm_resnet18.pth", map_location=DEVICE)
)
tnm_model.eval()

# -----------------------------
# Image Preprocessing
# -----------------------------
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# -----------------------------
# Load Image (replace path with your image)
# -----------------------------
image_path = "/kaggle/input/rsna-breast-cancer-detection/test/cancer/10432_1434858530.png"
image = Image.open(image_path).convert("RGB")
image_tensor = preprocess(image).unsqueeze(0).to(DEVICE)  # Add batch dimension

# -----------------------------
# Inference
# -----------------------------
with torch.no_grad():
    # Binary prediction
    binary_out = binary_model(image_tensor)
    binary_class = torch.argmax(binary_out, dim=1).item()
    binary_label = "Cancer" if binary_class == 1 else "Normal"

    # If cancer, predict TNM
    if binary_class == 1:
        T_pred, N_pred, M_pred = tnm_model(image_tensor)
        T = torch.argmax(T_pred, dim=1).item()
        N = torch.argmax(N_pred, dim=1).item()
        M = torch.argmax(M_pred, dim=1).item()
    else:
        T, N, M = None, None, None

# -----------------------------
# Results
# -----------------------------
print(f"Binary Prediction: {binary_label}")
if binary_class == 1:
    print(f"T Stage: {T}, N Stage: {N}, M Stage: {M}")


Binary Prediction: Cancer
T Stage: 1, N Stage: 0, M Stage: 1
